# 01 — Chunking experiments (Week 2, slice-first, optimized)

Notebook-only experiment on the frozen corpus. **No `src/`, LLD, taskboard, or Week 1 gold modifications.**

- Exploration grid (18): sizes **256/512/1024 tokens** (`len(tokenize(text))`, `tokenizer.v1`), overlaps **0/10%**, strategies **fixed / section / table**.
- Anchors (separate baselines, LLD-mandated): **C1** page-per-page 512/0; **C2** fixed 800/100 **char** windows.
- All 20 configs run on one fixed deterministic sample first; full corpus is opt-in and sequential. Results are actual measured outputs only.

**Optimization (no metric/semantic change):** document extraction, tokenization, spans, sections and evidence excerpts cached once; per-config chunking is single-pass and evidence containment reuses generated chunks for evidence pages only (no re-chunking/re-tokenization); efficient contiguous-token subsequence matching; sequential execution with per-config progress/timing and partial-result flush.


In [1]:
# Parameters (restart-and-run-all reproducible)
FREEZE_PATH = "data/corpus/freeze_v1.json"
PAGE_DIR = "data/corpus/extract/page_extracts"
TABLE_DIR = "data/corpus/extract/table_records"
GOLD_PATH = "data/eval/datasets/gold_v2_eval_v8_public.jsonl"
SAMPLE_TARGET = 16              # deterministic representative sample size
SAMPLE_SALT = "w2-chunk-v1"     # fixed; no randomness
RUN_SAMPLE_GRID = True
RUN_FULL_CORPUS_SELECTED = False
SELECTED_FOR_FULL = []          # e.g. ["C1", "C2", "table_tok512_ov10"]
RESULTS_PATH = "results/week2_chunking_experiments.json"
PARTIAL_RESULTS_PATH = "results/week2_chunking_experiments.partial.json"



In [2]:
import os, sys, json, hashlib, time, tracemalloc, re
from pathlib import Path
from collections import defaultdict

_root = Path.cwd().resolve()
if not (_root / "src").is_dir():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))

from src.rag_ingestion.tokenizer import tokenize, TOKENIZER_VERSION

def token_count(text: str) -> int:
    # Deterministic token count via shared tokenizer (LLD section 5). No shared count fn exists in src.
    return len(tokenize(text or ""))

FREEZE = json.loads(Path(FREEZE_PATH).read_text(encoding="utf-8"))
FREEZE_SHA = FREEZE.get("manifest_sha256")
print("tokenizer:", TOKENIZER_VERSION,
      "| deterministic:", token_count("Net Profit 1,23,456.78") == token_count("Net Profit 1,23,456.78"))
print("freeze manifest_sha256:", FREEZE_SHA)
print("corpus docs:", len(FREEZE.get("documents", [])))



tokenizer: tokenizer.v1 | deterministic: True
freeze manifest_sha256: 899ac206a760af531e102e01daf99e4ff61d36cb6fc1e2e7a28cf01b094b5edd
corpus docs: 1022


## LLD validation, mappings and recorded deviations

**Final read-only validation (passed):**
1. **Tokenizer** — `src/rag_ingestion/tokenizer.tokenize`, `TOKENIZER_VERSION=tokenizer.v1`; deterministic. No shared count function exists, so token count = `len(tokenize(text))` (notebook-local; no `src/` change, no new dependency).
2. **Table identity** — data has no `table_id`; mapping: `table_id -> table_logical_key` (stable logical identity); `table_extraction_id` retained as provenance. Field is `reporting_basis` (not `basis`). Canonical `chunk_id = sha256(json.dumps({"document_id","table_id","page_start","page_end"}, sort_keys=True, separators=(",",":")))`.
3. **Page boundaries / skipped pages** — page extracts carry `usable_as_evidence`, `page_class`, `pdf_page_index`; skipped counts computable per document. Index only `usable_as_evidence=True`.
4. **C1–C4** — confirmed in `lld_retrieval.txt` section 5.

**Deviations recorded (not silently applied):**
- **Window boundaries** use the tokenizer.v1 token *spans* (same regex) for speed; per-chunk size reported via `token_count()`; FY/quarter bare-year alternates add tokens to the reported count but not to span-based windowing.
- **Section-aware rules** are under-specified in the LLDs (only `section_context` exists on tables). A deterministic heading heuristic is used; exceptions and short sections are recorded.
- **Table-region exclusion from prose** cannot be exact without layout coordinates. Best-effort: prose keeps a table marker on table pages and cell/prose duplication is *measured* per config.
- **C1** is page-per-page (no merge, no split to enforce 512); pages >512 tokens are flagged.
- **search_text / display_text**: one derivation path (`display_text` = source text; `search_text` = identity header + display_text).

**Bottleneck (profiled, addressed):** the previous run re-chunked pages and re-tokenized chunks per evidence record and per config. Optimized by caching extraction/tokenization/spans/sections once and reusing generated evidence-page chunks for containment (same semantics).


In [3]:
# --- deterministic representative sample (identical across all configs) ---
PILOT = {"bhel", "maruti_suzuki", "deepak_nitrite", "srf", "berger_paints", "dabur", "bosch"}
pdir, tdir = Path(PAGE_DIR), Path(TABLE_DIR)

def _pages(doc_id):
    p = pdir / (doc_id + ".pages.jsonl")
    if not p.is_file():
        return []
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]

def _tables(doc_id):
    p = tdir / (doc_id + ".tables.jsonl")
    if not p.is_file() or p.stat().st_size == 0:
        return []
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]

def _has_table(doc_id):
    p = tdir / (doc_id + ".tables.jsonl")
    return p.is_file() and p.stat().st_size > 0

docs = []
for d in FREEZE["documents"]:
    pg = _pages(d["document_id"])
    usable = sum(1 for r in pg if r.get("usable_as_evidence"))
    if usable == 0:
        continue
    docs.append({**d, "_n_pages": len(pg), "_usable": usable, "_has_table": _has_table(d["document_id"]),
                 "_pilot": d["company_id"] in PILOT})

gold = [json.loads(l) for l in Path(GOLD_PATH).read_text(encoding="utf-8").splitlines() if l.strip()]
dev_answer = [r for r in gold if r.get("split") == "dev" and r.get("expected_behaviour") == "answer"]
ev_docs = set()
for r in dev_answer:
    for e in (r.get("evidence") or []):
        if isinstance(e, dict) and e.get("document_id"):
            ev_docs.add(e["document_id"])

def hkey(doc_id):
    return hashlib.sha256((SAMPLE_SALT + doc_id).encode()).hexdigest()

by_hash = sorted(docs, key=lambda d: hkey(d["document_id"]))
selected, chosen = [], set()

def take(pred, want):
    n = 0
    for d in by_hash:
        if len(selected) >= SAMPLE_TARGET or n >= want:
            break
        if d["document_id"] in chosen or not pred(d):
            continue
        selected.append(d); chosen.add(d["document_id"]); n += 1
    return n

take(lambda d: d["document_id"] in ev_docs, 6)      # gold-evidence coverage
take(lambda d: d["_pilot"], 4)                       # pilot companies
take(lambda d: d["_has_table"] and d["doc_type"] == "annual", 3)
take(lambda d: d["_has_table"] and d["doc_type"] == "quarterly", 3)
take(lambda d: d["doc_type"] == "annual", 2)
take(lambda d: d["doc_type"] == "quarterly", 2)
take(lambda d: True, SAMPLE_TARGET)                  # fill

selected.sort(key=lambda d: d["document_id"])
SAMPLE_IDS = [d["document_id"] for d in selected]
SAMPLE = {d["document_id"]: d for d in selected}

print("sample docs:", len(SAMPLE_IDS))
print("pilot:", sum(1 for d in selected if d["_pilot"]), "| annual:", sum(1 for d in selected if d["doc_type"] == "annual"),
      "| quarterly:", sum(1 for d in selected if d["doc_type"] == "quarterly"),
      "| with tables:", sum(1 for d in selected if d["_has_table"]),
      "| in dev evidence:", sum(1 for d in selected if d["document_id"] in ev_docs))
print("sample_ids:", SAMPLE_IDS)



sample docs: 16
pilot: 5 | annual: 9 | quarterly: 6 | with tables: 16 | in dev evidence: 6
sample_ids: ['0752897a3e1912c82839aeabce5695c269395e02d1dee6a70da992bb2c6c474c', '0a50625874bd7177af19d5f7abcaa86c3d6f124a48f4ac2f44ca06c5fe8a5897', '0ea899ba19c2727c06fc6926b1a4180c8edbecdde684e15a48fb49267c7ffc81', '1561c2cc0a05c3a7dd712e5cfa28a7538315b74f73103964fc24f85d89a80158', '224f0692e4bdbd95a489b85664faeb2c2ad7a948bcfce82d58da0216d9726c63', '263eb5278846cba9952a2116cb07ec5e6c5b16990cd91b310ab595a6d2afe133', '3136cbe842dbea7de663751fb2288a9c7ad0c98b0ee712ddc6004a34344b0375', '437973dd06235c9dbf03b1aea9d85a2d4335ccf301f3f259900d25350a153342', '5d003d73d805ab187524fd793493db1ac1696c72a2d66f1c7eabcb89bc618788', '87ca5fadbfffea49ab58395c2998abbeac8a7d9a872a423ca7acf88f3ae4092b', '8f08c8d35f90f77ded528dc3f16e6a05c6fdefee9f1c5432eeec3838b83a2ea0', 'ca0e00449b6ce124ccec850c03a121e1f920a5d51f229074571083dfe6a1288e', 'dc62cdc1cef21c160d8fd9acbc0b8c0f8f628a4dcb215a0e93e5e366d6382740', 'e0a61554d69

In [4]:
# --- 20 configs: 18 exploration grid + 2 LLD anchors ---
GRID_SIZES = (256, 512, 1024)
GRID_OVERLAPS = (0.0, 0.10)
STRATEGIES = ("fixed", "section", "table")

CONFIGS = {}
for strat in STRATEGIES:
    for size in GRID_SIZES:
        for ov in GRID_OVERLAPS:
            CONFIGS[f"{strat}_tok{size}_ov{int(round(ov*100))}"] = {
                "strategy": strat, "unit": "tokens", "size": size, "overlap_frac": ov,
                "anchor": False, "outside_lld_spec": False}
CONFIGS["C1"] = {"strategy": "page_per_page", "unit": "tokens", "size": 512, "overlap_frac": 0.0,
                 "anchor": True, "outside_lld_spec": False}
CONFIGS["C2"] = {"strategy": "fixed_char", "unit": "chars", "size": 800, "overlap": 100,
                 "anchor": True, "outside_lld_spec": False}

print("total configs:", len(CONFIGS), "| grid:", len(CONFIGS) - 2, "| anchors: C1, C2")



total configs: 20 | grid: 18 | anchors: C1, C2


In [5]:
# --- helpers: token spans, section detection (offset-aware), windows, tables, subsequence ---
_TOK_RE = re.compile(r"\(?-?\d[\d,]*(?:\.\d+)?\)?|[a-z0-9]+", re.IGNORECASE)

def tok_spans(text):
    return [m.span() for m in _TOK_RE.finditer(text or "")]

HEADING_MAXLEN = 80
STMT_KEYS = ("balance sheet", "profit and loss", "income statement", "cash flow",
             "statement of", "notes to", "segment", "equity", "financial statements")

def is_heading(line: str) -> bool:
    t = line.strip()
    if not t or len(t) > HEADING_MAXLEN:
        return False
    if t.isupper() and any(c.isalpha() for c in t):
        return True
    low = t.lower()
    if any(k in low for k in STMT_KEYS) and len(t.split()) <= 8:
        return True
    if t.endswith(":") and len(t.split()) <= 10:
        return True
    words = t.split()
    if 1 <= len(words) <= 8 and not any(ch.isdigit() for ch in t):
        caps = sum(1 for w in words if w[:1].isupper())
        if caps >= max(1, len(words) - 1):
            return True
    return False

def detect_sections(text: str):
    # returns list of (head, body, body_start, body_end) with offsets in original text
    text = text or ""
    lines = []  # (line_text, start, end)
    i = 0
    for part in text.split("\n"):
        s = i
        e = s + len(part)
        lines.append((part, s, e))
        i = e + 1  # skip newline
    blocks = []
    cur_head = None
    cur_body_s = 0
    cur_body_e = 0
    have = False

    def flush(h, bs, be):
        if bs is None or be is None or be < bs:
            bs, be = 0, 0
        blocks.append((h, text[bs:be], bs, be))

    for ln, s, e in lines:
        if is_heading(ln):
            if have:
                flush(cur_head, cur_body_s, cur_body_e)
            cur_head = ln.strip()
            cur_body_s = min(e + 1, len(text))
            cur_body_e = cur_body_s
            have = True
        else:
            if not have:
                cur_head = None
                cur_body_s = s
                cur_body_e = e
                have = True
            else:
                cur_body_e = e
    if have:
        flush(cur_head, cur_body_s, cur_body_e)
    elif text:
        blocks.append((None, text, 0, len(text)))
    return blocks

def windows_by_spans(text, spans, size, overlap_frac):
    # returns list of (s_char, e_char, tok_i, tok_j)
    if not spans:
        return []
    step = max(1, size - int(round(size * overlap_frac)))
    out, i = [], 0
    n = len(spans)
    while i < n:
        j = min(n, i + size)
        s, e = spans[i][0], spans[j - 1][1]
        if e > s:
            out.append((s, e, i, j))
        if j >= n:
            break
        i += step
    return out

def windows_by_chars(text, size, overlap):
    step = max(1, size - overlap)
    out, i = [], 0
    n = len(text)
    while i < n:
        s, e = i, min(n, i + size)
        if text[s:e].strip():
            out.append((s, e, None, None))
        if e >= n:
            break
        i += step
    return out

# --- table rendering per LLD section 5 ---
def render_table(rec):
    parts = []
    cap = (rec.get("caption") or "").strip()
    if cap:
        parts.append("caption: " + cap)
    parts.append("statement_kind: " + str(rec.get("statement_kind") or ""))
    uh = (rec.get("unit_header") or "").strip()
    if uh:
        parts.append("unit_header: " + uh)
    parts.append("reporting_basis: " + str(rec.get("reporting_basis") or ""))
    cols = rec.get("columns") or []
    if cols:
        parts.append("columns: " + " | ".join(
            f"{c.get('column_id')}:{c.get('header_raw')}" + (f" <{c.get('header_path')}>" if c.get("header_path") else "")
            for c in cols))
    ordmap = {c.get("column_id"): (c.get("column_ordinal") if isinstance(c.get("column_ordinal"), int) else i)
              for i, c in enumerate(cols)}
    for row in rec.get("rows") or []:
        label = (row.get("row_label_raw") or "").strip()
        pairs = []
        for cell in sorted(row.get("cells") or [], key=lambda x: ordmap.get(x.get("column_id"), 0)):
            raw = str(cell.get("raw") or "").strip()
            if raw:
                pairs.append(f"{cell.get('column_id')}={raw}")
        if label and pairs:
            parts.append(label + ": " + "; ".join(pairs))
        elif pairs:
            parts.append("; ".join(pairs))
    return "\n".join(p for p in parts if p.strip())

def table_id_of(rec):
    return rec.get("table_logical_key") or rec.get("table_extraction_id") or hashlib.sha256(
        f"{rec.get('document_id')}|{rec.get('page_start')}|{rec.get('statement_kind')}|{(rec.get('caption') or '')}".encode()
    ).hexdigest()

def table_chunk_id(rec):
    payload = json.dumps({"document_id": rec.get("document_id"), "table_id": table_id_of(rec),
                          "page_start": rec.get("page_start"), "page_end": rec.get("page_end")},
                         sort_keys=True, separators=(",", ":"))
    return "table:" + hashlib.sha256(payload.encode()).hexdigest()

def identity_header(doc, statement_kind=""):
    return (f"Company: {doc.get('company_id')} | Period: {doc.get('fiscal_year')} | "
            f"Basis: {doc.get('variant')} | Doc type: {doc.get('doc_type')} | {statement_kind}")

# contiguous token-subsequence match (same semantics), optimized
def has_subseq(hay, needle):
    n = len(needle)
    m = len(hay)
    if n == 0 or n > m:
        return False
    if n == 1:
        return needle[0] in hay
    first = needle[0]
    last = needle[-1]
    i = 0
    lim = m - n + 1
    while i < lim:
        try:
            j = hay.index(first, i)
        except ValueError:
            return False
        if j >= lim:
            return False
        if hay[j + n - 1] == last and hay[j:j + n] == needle:
            return True
        i = j + 1
    return False



In [6]:
# --- build caches once (extraction, tokenization, spans, sections, evidence) + profile ---
def _tok_list_from_spans(text, spans):
    return [text[a:b] for a, b in spans]

t_phase = {}
_t0 = time.time(); tracemalloc.start()

CACHE = {}  # doc_id -> {pages: {page: {...}}, tables: [...]}
for doc_id in SAMPLE_IDS:
    pages = {}
    for rec in _pages(doc_id):
        if not rec.get("usable_as_evidence"):
            continue
        text = rec.get("page_text") or ""
        page = rec.get("pdf_page_index")
        spans = tok_spans(text)
        toks = _tok_list_from_spans(text, spans)
        secs = []
        for head, body, bs, be in detect_sections(text):
            btext = body if (be > bs) else ""
            b_spans = tok_spans(btext)
            b_toks = _tok_list_from_spans(btext, b_spans)
            secs.append({"head": head, "body": btext, "bs": bs, "be": be, "spans": b_spans, "toks": b_toks})
        pages[page] = {"text": text, "spans": spans, "toks": toks, "sections": secs}
    tables = _tables(doc_id)
    # pre-render/tokenize table chunks (used by table configs)
    table_chunks = []
    for tr in tables:
        ttext = render_table(tr)
        if not ttext.strip():
            continue
        ttoks = tuple(tokenize(ttext))
        table_chunks.append({"rec": tr, "text": ttext, "toks": ttoks, "tc": len(ttoks)})
    CACHE[doc_id] = {"pages": pages, "tables": table_chunks}

t_phase["load_tokenize_sections_sec"] = round(time.time() - _t0, 3)

# evidence records (sample docs only) and pre-tokenized excerpts
EV = []
for r in dev_answer:
    for e in (r.get("evidence") or []):
        if isinstance(e, dict) and e.get("document_id") in SAMPLE and isinstance(e.get("pdf_page"), int):
            etoks = tuple(tokenize(e.get("excerpt") or ""))
            EV.append((e["document_id"], e["pdf_page"], etoks))
EV_PAGES = {(d, p) for d, p, _ in EV}

_cur, _peak = tracemalloc.get_traced_memory()
t_phase["cache_peak_mem_bytes"] = _peak
print("cache built: docs", len(CACHE),
      "| pages", sum(len(v["pages"]) for v in CACHE.values()),
      "| table_chunks", sum(len(v["tables"]) for v in CACHE.values()))
print("evidence records (sample):", len(EV), "| evidence pages:", len(EV_PAGES))
print("phase/profile:", t_phase)



cache built: docs 16 | pages 2659 | table_chunks 5655
evidence records (sample): 11 | evidence pages: 6
phase/profile: {'load_tokenize_sections_sec': 291.119, 'cache_peak_mem_bytes': 787442522}


In [7]:
# --- per-config runner (optimized, sequential, containment reuses chunks) ---
def _boundary_number_split(text, s, e):
    left = text[s - 1] if s > 0 else ""
    right = text[e] if e < len(text) else ""
    first = text[s:s + 1]
    last = text[e - 1:e]
    a = first.isdigit() and (left.isdigit() or left in ",.")
    b = last.isdigit() and (right.isdigit() or right in ",.")
    return int(a or b)

def _merge_len(intervals):
    if not intervals:
        return 0
    intervals.sort()
    u, cs, ce = 0, intervals[0][0], intervals[0][1]
    for s, e in intervals[1:]:
        if s > ce:
            u += ce - cs
            cs, ce = s, e
        else:
            if e > ce:
                ce = e
    u += ce - cs
    return u

def prose_pieces_for_page(cfg, page_entry):
    # yields (s, e, tok_count, tok_i, tok_j, is_section_body_short)
    text = page_entry["text"]
    spans = page_entry["spans"]
    strat = cfg["strategy"]
    if strat == "page_per_page":
        yield (0, len(text), len(spans), 0, len(spans), False)
        return
    if strat == "fixed_char":
        for s, e, ti, tj in windows_by_chars(text, cfg["size"], cfg["overlap"]):
            tc = token_count(text[s:e])
            yield (s, e, tc, None, None, False)
        return
    if strat == "section":
        size = cfg["size"]; ov = cfg["overlap_frac"]
        for sec in page_entry["sections"]:
            btext = sec["body"]; bs = sec["bs"]; be = sec["be"]
            if not btext.strip():
                continue
            b_spans = sec["spans"]; b_toks_len = len(b_spans)
            if b_toks_len < size:
                yield (bs, be, b_toks_len, None, None, True)
            else:
                for ls, le, ti, tj in windows_by_spans(btext, b_spans, size, ov):
                    yield (bs + ls, bs + le, tj - ti, None, None, False)
        return
    # fixed (token windows) and table strategy prose: token windows over page
    for s, e, ti, tj in windows_by_spans(text, spans, cfg["size"], cfg["overlap_frac"]):
        yield (s, e, tj - ti, ti, tj, False)

def run_config(name, cfg, doc_ids):
    t0 = time.perf_counter()
    tracemalloc.start()
    n_chunks = 0
    tok_sum = 0
    unique_chars = 0
    number_splits = 0
    short_sections = 0
    table_records = 0
    table_fragmented = 0
    table_cells = set()
    prose_tokens = set()
    ev_chunk_toks = defaultdict(list)  # (doc,page) -> [tok_tuple,...] for evidence pages only

    for doc_id in doc_ids:
        doc_cache = CACHE[doc_id]
        doc_intervals = []
        for page, page_entry in doc_cache["pages"].items():
            text = page_entry["text"]
            page_toks = page_entry["toks"]
            need_ev = (doc_id, page) in EV_PAGES
            for s, e, tc, ti, tj, is_short in prose_pieces_for_page(cfg, page_entry):
                if e <= s:
                    continue
                n_chunks += 1
                tok_sum += tc
                doc_intervals.append((s, e))
                number_splits += _boundary_number_split(text, s, e)
                if is_short:
                    short_sections += 1
                # tokens for this chunk (for ev containment and table-prose-dup)
                if ti is not None and tj is not None:
                    ctoks = tuple(page_toks[ti:tj])
                else:
                    ctoks = tuple(tokenize(text[s:e]))
                if need_ev:
                    ev_chunk_toks[(doc_id, page)].append(ctoks)
                if cfg["strategy"] == "table":
                    prose_tokens.update(ctoks)
        unique_chars += _merge_len(doc_intervals)

        if cfg["strategy"] == "table":
            for tch in doc_cache["tables"]:
                rec = tch["rec"]
                table_records += 1
                n_chunks += 1
                tok_sum += tch["tc"]
                if rec.get("page_start") != rec.get("page_end"):
                    table_fragmented += 1
                for row in (rec.get("rows") or []):
                    for c in (row.get("cells") or []):
                        v = str(c.get("raw") or "").strip()
                        if v:
                            table_cells.add(v)

    # evidence containment (same semantics): contiguous excerpt token seq in any chunk of that (doc,page)
    if EV:
        hits = 0
        for doc_id, page, etoks in EV:
            chunks = ev_chunk_toks.get((doc_id, page))
            if not chunks:
                continue
            found = False
            if etoks:
                for ct in chunks:
                    if has_subseq(ct, etoks):
                        found = True
                        break
            hits += int(found)
        evidence_containment = hits / len(EV)
    else:
        evidence_containment = None

    dup_cells = 0
    if cfg["strategy"] == "table" and table_cells:
        for v in table_cells:
            vt = set(tokenize(v))
            if vt and vt.issubset(prose_tokens):
                dup_cells += 1

    cur, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    elapsed = time.perf_counter() - t0
    return {
        "config": name, **{k: v for k, v in cfg.items()},
        "chunk_count": n_chunks,
        "mean_tokens": (tok_sum / n_chunks) if n_chunks else 0.0,
        "total_tokens": tok_sum,
        "unique_chars": unique_chars,
        "number_splits": number_splits,
        "short_sections": short_sections,
        "table_records": table_records,
        "table_fragmented": table_fragmented,
        "table_cell_prose_duplicates": dup_cells,
        "evidence_containment": evidence_containment,
        "peak_mem_bytes": peak,
        "seconds": round(elapsed, 4),
        "n_docs": len(doc_ids),
    }



In [8]:
# --- run all 20 configs sequentially with progress/timing and partial flush ---
RESULTS = {}
RUN_META = {"completed": [], "failed": [], "all_complete": False}
_t_run = time.perf_counter()
_peak_run = 0

def _flush_partial():
    out = {
        "experiment": "week2_chunking",
        "status": "partial" if not RUN_META["all_complete"] else "complete",
        "corpus_manifest_sha256": FREEZE_SHA,
        "tokenizer_version": TOKENIZER_VERSION,
        "sample_salt": SAMPLE_SALT,
        "sample_ids": SAMPLE_IDS,
        "configs": CONFIGS,
        "results": RESULTS,
        "run_meta": RUN_META,
    }
    Path(PARTIAL_RESULTS_PATH).parent.mkdir(parents=True, exist_ok=True)
    Path(PARTIAL_RESULTS_PATH).write_text(json.dumps(out, indent=2, sort_keys=True), encoding="utf-8")

if RUN_SAMPLE_GRID:
    names = list(CONFIGS.keys())
    for idx, name in enumerate(names, 1):
        cfg = CONFIGS[name]
        print(f"[{idx}/{len(names)}] START {name} (strategy={cfg['strategy']} size={cfg.get('size')} ov={cfg.get('overlap_frac', cfg.get('overlap'))})", flush=True)
        try:
            res = run_config(name, cfg, SAMPLE_IDS)
            RESULTS[name] = res
            RUN_META["completed"].append(name)
            _peak_run = max(_peak_run, res.get("peak_mem_bytes", 0))
            print(f"[{idx}/{len(names)}] DONE  {name} chunks={res['chunk_count']} mean_tok={res['mean_tokens']:.1f} "
                  f"uniq={res['unique_chars']} splits={res['number_splits']} tbl={res['table_records']} "
                  f"ev={res['evidence_containment']} sec={res['seconds']}", flush=True)
        except Exception as e:
            RUN_META["failed"].append({"config": name, "error": repr(e)})
            print(f"[{idx}/{len(names)}] FAIL  {name} err={e!r}", flush=True)
            _flush_partial()
            break
        _flush_partial()
    RUN_META["all_complete"] = (len(RESULTS) == len(CONFIGS)) and not RUN_META["failed"]
else:
    print("RUN_SAMPLE_GRID=False")

RUN_META["total_seconds"] = round(time.perf_counter() - _t_run, 4)
RUN_META["peak_mem_bytes_observed"] = _peak_run
print("run summary:", json.dumps(RUN_META, sort_keys=True))



[1/20] START fixed_tok256_ov0 (strategy=fixed size=256 ov=0.0)


[1/20] DONE  fixed_tok256_ov0 chunks=6987 mean_tok=206.2 uniq=96094 splits=1664 tbl=0 ev=0.0 sec=4.1291


[2/20] START fixed_tok256_ov10 (strategy=fixed size=256 ov=0.1)


[2/20] DONE  fixed_tok256_ov10 chunks=7350 mean_tok=212.6 uniq=96100 splits=1670 tbl=0 ev=0.0 sec=1.7277


[3/20] START fixed_tok512_ov0 (strategy=fixed size=512 ov=0.0)


[3/20] DONE  fixed_tok512_ov0 chunks=4109 mean_tok=350.6 uniq=96097 splits=1543 tbl=0 ev=0.0 sec=0.189


... (31 outputs left out)

[19/20] DONE  C1 chunks=2659 mean_tok=541.8 uniq=96104 splits=1485 tbl=0 ev=0.0 sec=0.1015


[20/20] START C2 (strategy=fixed_char size=800 ov=100)


[20/20] DONE  C2 chunks=14673 mean_tok=111.1 uniq=96104 splits=2444 tbl=0 ev=1.0 sec=32.4817


run summary: {"all_complete": true, "completed": ["fixed_tok256_ov0", "fixed_tok256_ov10", "fixed_tok512_ov0", "fixed_tok512_ov10", "fixed_tok1024_ov0", "fixed_tok1024_ov10", "section_tok256_ov0", "section_tok256_ov10", "section_tok512_ov0", "section_tok512_ov10", "section_tok1024_ov0", "section_tok1024_ov10", "table_tok256_ov0", "table_tok256_ov10", "table_tok512_ov0", "table_tok512_ov10", "table_tok1024_ov0", "table_tok1024_ov10", "C1", "C2"], "failed": [], "peak_mem_bytes_observed": 787442522, "total_seconds": 135.0257}


In [9]:
# --- save final results (only if all 20 complete) + output validation ---
assert RUN_META.get("all_complete"), f"incomplete: completed={len(RUN_META.get('completed',[]))} failed={RUN_META.get('failed')}"
assert len(RESULTS) == 20, f"expected 20 configs, got {len(RESULTS)}"

out = {
    "experiment": "week2_chunking",
    "status": "complete",
    "corpus_manifest_sha256": FREEZE_SHA,
    "tokenizer_version": TOKENIZER_VERSION,
    "tokenizer_count_method": "len(tokenize(text)) via src.rag_ingestion.tokenizer.tokenize",
    "sample_salt": SAMPLE_SALT,
    "sample_ids": SAMPLE_IDS,
    "sample_n_docs": len(SAMPLE_IDS),
    "configs": CONFIGS,
    "results": RESULTS,
    "run_meta": RUN_META,
}
Path(RESULTS_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(RESULTS_PATH).write_text(json.dumps(out, indent=2, sort_keys=True), encoding="utf-8")

# validation
obj = json.loads(Path(RESULTS_PATH).read_text(encoding="utf-8"))
req_top = ["corpus_manifest_sha256", "tokenizer_version", "sample_ids", "configs", "results"]
missing = [k for k in req_top if k not in obj]
metric_keys = {"chunk_count","mean_tokens","total_tokens","unique_chars","number_splits","short_sections","table_records","table_fragmented","table_cell_prose_duplicates","evidence_containment","peak_mem_bytes","seconds","n_docs"}
bad = [n for n,r in obj["results"].items() if not metric_keys.issubset(r.keys())]
print("wrote", RESULTS_PATH)
print("configs_in_results:", len(obj["results"]), "| missing_top:", missing, "| configs_missing_metrics:", bad)
print("manifest_sha256:", obj["corpus_manifest_sha256"], "| tokenizer:", obj["tokenizer_version"])
print("sample_ids match:", obj["sample_ids"] == SAMPLE_IDS)



wrote results/week2_chunking_experiments.json
configs_in_results: 20 | missing_top: [] | configs_missing_metrics: []
manifest_sha256: 899ac206a760af531e102e01daf99e4ff61d36cb6fc1e2e7a28cf01b094b5edd | tokenizer: tokenizer.v1
sample_ids match: True


## Reading the results (no interpretation is invented here)

- Compare `chunk_count`, `mean_tokens`, `unique_chars` (overlap counted once), `number_splits`, `table_records`, `table_fragmented`, `table_cell_prose_duplicates`, `evidence_containment`, `seconds`, `peak_mem_bytes` per config.
- C1/C2 are anchors, not grid entries.
- Scaling: set `SELECTED_FOR_FULL` and `RUN_FULL_CORPUS_SELECTED=True` below to run chosen configs over the full frozen corpus (sequential).


In [10]:
# --- optional: full-corpus run of selected configs (sequential, opt-in) ---
FULL_RESULTS = {}
if RUN_FULL_CORPUS_SELECTED and SELECTED_FOR_FULL:
    full_ids = []
    for d in FREEZE["documents"]:
        pg = _pages(d["document_id"])
        if any(r.get("usable_as_evidence") for r in pg):
            full_ids.append(d["document_id"])
    full_ids.sort()
    print("full-corpus docs:", len(full_ids), "- note: this cell does not reuse the sample cache; may be slow")
    print("skipped: implement full-corpus cache if/when needed")
else:
    print("full-corpus run skipped (opt-in)")



full-corpus run skipped (opt-in)
